# Country Factor (USA) — Construction Spec (USE4-faithful)

**Purpose.** Source-of-truth spec for the `country` build. The country factor
is the **intercept of the cross-sectional regression**: every stock has
exposure exactly 1. There is nothing to estimate on the exposure side — the
deliverable's real content is (a) the **anchor**: the per-date regression
universe with its √mcap regression weights, and (b) a **validation CSR** run
over every month-transition proving the full system (country + 55 industries +
12 styles, cap-weighted industry constraint) assembles into a well-posed,
economically correct regression.

✅ **PDF SPEC (USE4 Methodology Notes §2.1; Empirical Notes §4).** USE4 adds a
Country factor to the USE3 structure: *every asset has unit exposure to it*,
industry factor returns are constrained so their cap-weighted sum is zero, and
the country factor return is reported to track the cap-weighted estimation
universe return with ≈ 99.9% correlation. The industry factors thereby become
pure relative (industry-minus-market) bets.

**Identification.** With an intercept and one 0/1 industry membership per
stock, the 55 industry columns sum to the country column — X′X is singular.
USE4 resolves this with the linear constraint

```
Σ_j  w_jt · f_ind,j,t  =  0          w_jt = cap weight of industry j in ESTU at t
```

implemented here by exact reparametrization (restricted least squares): one
industry's factor return is expressed from the other 54 via the constraint,
the reduced system is solved by WLS, and the eliminated factor return is
recovered. The constraint weights are the shipped
`industry_weights_use4.parquet` (built and validated in step `industries`).

> **Your task.** The spec is provided (the CSR and overview chapters in `docs/textbook/` carry the reference numbers); `country_build.ipynb` is yours to write. **This step closes the exposure half of the pipeline** — its validation CSR consumes your style factors and your industry factors, so finish steps 02–03 first; the risk-model stages (05–08) then build on its anchor.

## §1. Deliverables and output schema

**Anchor (the model deliverable):** `data/out/country_use4.parquet`,
compression zstd, statistics=True.

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date |
| `in_estu` | Bool | Always True — the anchor *is* the regression universe |
| `mcap` | Float64 | Market cap on `signal_date` |
| `country` | Float64 | Country exposure — always exactly 1.0 |
| `w_reg` | Float64 | √mcap regression weight, normalized to sum to 1 per date |

One row per in-ESTU stock-date (822,918 rows over 330 dates on the reference
run).

**Validation artifact (evidence, NOT a model deliverable):**
`data/out/csr_validation_returns.parquet` — the validation CSR's factor
returns, one row per (return month × factor):

| Column | Type | Description |
|---|---|---|
| `signal_date` | Date | Exposure date t (start of the return month) |
| `ret_date` | Date | End of the return month (the next signal date) |
| `factor` | String | `country`, the 55 industry names, or the 12 style names |
| `f` | Float64 | Factor return for the month (null if the column was degenerate that month) |
| `r2` | Float64 | Weighted cross-sectional R² of the month's regression (repeated per row) |
| `n_stocks` | UInt32 | Stocks in the month's regression (repeated per row) |

The real factor-return deliverable belongs to the future CSR module; this file
exists so the validation CSR is reproducible and so the production CSR (step `05_csr`) has a
reference series to reconcile against.

## §2. Pipeline

```
STAGE 0  Setup, parameters
STAGE 1  Anchor: in-ESTU universe -> country exposure + normalized sqrt(mcap)
         weights -> save country_use4.parquet
STAGE 2  Monthly returns: compound daily excess log returns over each
         signal-month (stock and benchmark) via the shared daily artifact
STAGE 3  Exposure matrix: join the 12 style scores + industry labels +
         constraint weights onto the anchor
STAGE 4  Validation CSR: per month-transition, constrained WLS
         (country + 55 industries + 12 styles, sqrt(mcap) weights) ->
         save csr_validation_returns.parquet
STAGE 5  Validate
```

**PIT discipline:** exposures dated t explain returns over (t, t+1] — the
regression uses only information known at t.

**Position in the pipeline:** the anchor needs only `estu_monthly.parquet`,
but the validation CSR consumes every exposure deliverable — `country`
therefore runs **last**, after every style and industry build.

## §3. STAGE 0 — Parameters

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
# ✅ PDF SPEC — country exposure is definitional
COUNTRY_EXPOSURE = 1.0

# ✅ PDF SPEC — WLS weights proportional to sqrt(mcap) (idio variance ~ 1/sqrt(mcap))
#    Constraint: cap-weighted industry factor returns sum to zero per date.

# SUGGESTION (NOT IN PDF) — validation tolerances
FC_CORR_MIN      = 0.99    # corr(f_c, monthly benchmark excess return)
CONSTRAINT_TOL   = 1e-10   # max |sum_j w_j f_ind_j| over dates
R2_MEAN_MIN      = 0.15    # mean weighted cross-sectional R^2 floor
COND_MAX         = 1e6     # max condition number of the scaled design
RET_MISS_MAX     = 0.02    # max share of ESTU stocks without a monthly return
```

**Complete-case regression.** This build estimates nothing — the exposure is
1 by definition. Stocks missing any style or industry exposure on a date are
dropped from that month's regression and counted; expect the drops to
concentrate in young listings and the earliest dates. Report the share.

## §4. STAGE 4 — Constrained WLS (the heart of the build)

Per month-transition (t, t+1], with n stocks:

- **y** — excess log return compounded over the trading days owned by t
  (the shared `_td_to_sig` mapping; partial months from delistings kept as-is).
- **X = [1 | D | S]** — country column, 55 industry dummies (one 0/1 per
  stock), 12 standardized style scores.
- **v** — √mcap at t, normalized.

**NOT IN PDF — constraint implementation.** USE4 states the constraint, not
the algorithm.

**SUGGESTION — exact reparametrization.** Eliminate the **largest-cap-weight
industry e** at each date (numerically safest divisor): for j ≠ e the factor
return is free; `f_e = -(Σ_{j≠e} w_j f_j) / w_e`. Columns transform as
`D̃_j = D_j - (w_j / w_e) · D_e`, the reduced system `[1 | D̃ | S]` is solved
by WLS (√v-scaled `lstsq`), and `f_e` is recovered. This is exact restricted
least squares — the choice of e changes nothing but conditioning.

**NOT IN PDF — factor-dates without coverage.** gro, beta and nlb have no
scored stocks on a handful of early-1999 dates (annual-data and benchmark
warm-ups).

**SUGGESTION.** Drop a style column from the regression on any date where it
has no usable exposures; record `f = null` for that factor-date. Industries
are asserted non-degenerate (all 55 present at every date — validated
upstream).

**NOT IN PDF — return convention.** Monthly horizon is implied; the exact
compounding is not specified.

**SUGGESTION.** Sum of daily excess **log** returns over the owned trading
days — consistent with every time-series factor in this pipeline. Stocks with
no trading days in the month (delisted at t) are dropped from that month's
regression and counted.

**NOT IN PDF — unclassifiable stocks.** The industries panel labels stocks
its fallback ladder could not classify (`UNASSIGNED (fallback ladder)`); they
carry no industry factor and no constraint weight.

**SUGGESTION.** Drop them from the CSR sample, counted and gated tiny
(5 stock-dates / 1 ticker on the reference run; `UNASSIGNED_MAX = 50`).

**NOT IN PDF — risk-free publication lag.** The Ken French RF series
publishes with a lag, so the daily artifact's excess returns are null for the
most recent weeks — whole months at the calendar tail have no usable y.

**SUGGESTION.** Skip transitions with no excess-return data (gated: ≤ 3, all
within the last three calendar months). They are picked up on the next data
refresh. On the reference run: 2 skipped (2026-04-30, 2026-05-29 starts),
leaving 327 of 329 transitions.

## §5. STAGE 5 — Validation contract

| check | gate |
|---|---|
| 1 | corr(f_c, monthly benchmark excess return) > 0.99 — the USE4 signature (they report ≈ 0.999) |
| 2 | max over dates of \|Σ_j w_j f_ind,j\| < 1e-10 — the constraint holds to machine precision |
| 3 | mean weighted cross-sectional R² > 0.15 (informational detail: full distribution) |
| 4 | max condition number of the √v-scaled design < 1e6 — the reparametrized system is well-posed everywhere |
| 5 | dropped factor-dates ⊆ the expected early-1999 warm-up set (gro/beta/nlb), count ≤ 9 |
| 6 | anchor integrity: rows = in-ESTU count; keys unique; `country` ≡ 1.0; per-date Σ`w_reg` = 1 to 1e-12 |
| 7a | exposure-incomplete (complete-case) drops counted and reported per date |
| 7b | max monthly return-missing share < 2% |
| 7c | UNASSIGNED-industry stock-dates dropped ≤ 50 |
| 8 | skipped transitions ≤ 3, all within the RF-lag tail (last three calendar months) |

Plus informational prints: f_c annualized mean/vol vs the benchmark, RMSE of
(f_c − benchmark), R² distribution, eliminated-industry frequency, CSR sample
sizes.

**Shared validation conventions (all factors, 2026-06-10):** the final
in-progress signal date has no completed forward month; together with the
RF-publication lag this leaves 327 of the 329 month-transitions in the
validation CSR on the reference run. The anchor still covers all 330 exposure
dates.

## §6. Master list of NOT-IN-PDF decisions

| # | Decision | Suggestion | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Constraint implementation | Exact reparametrization, eliminating the largest-cap-weight industry per date | Lagrangian / pseudo-inverse on the augmented system | Never — algebraically identical; this is the numerically safest exact form |
| 2 | Regression weights | √mcap (USE4's idio-variance model), normalized per date; shipped as `w_reg` in the anchor | Equal weights; full mcap weights | If the CSR module adopts a different idio-variance model |
| 3 | Return convention | Sum of daily excess log returns over owned trading days; partial months kept | Simple-return compounding; drop partial months | If the CSR module standardizes differently — reconcile against the reference series |
| 4 | Factor-dates without coverage | Drop the style column that date, f = null | Drop the whole date | Never — dropping the date discards 60+ valid factor returns to avoid 1 null |
| 5 | Eliminated industry | argmax cap weight per date | Fixed industry across dates | Never — exact algebra either way |
| 6 | Validation artifact | Save `csr_validation_returns.parquet` as reproducible evidence (not a model deliverable) | Keep validation in-notebook only | When the CSR module ships its own factor returns — keep for reconciliation, then retire. The production CSR now lives in `05_csr/csr_spec.ipynb` and uses stricter sample rules (complete-case transitions with n < 100 stocks are skipped), so its transition count (~303) is by design lower than this validation CSR's 327 |
| 7 | Unclassifiable stocks | Drop `UNASSIGNED (fallback ladder)` rows from the CSR sample, counted and gated (≤ 50; 5 observed) | Assign to a synthetic 56th industry | If the unassigned population ever grows past the gate |
| 8 | RF publication lag | Skip transitions with no excess returns (≤ 3, RF-lag tail only); picked up on the next data refresh | Fall back to raw returns for the tail | Never — mixing raw and excess returns in one regression series is worse than a short tail |